# FloodA5 notebook 4 — the sub-grid (SGS) model on a strip with known answers

Up to FloodA5 0.6.0 the sub-grid (SGS) model stored far more water on the
Carlisle flood than the other models on the same A5 mesh: about 2.5 times
the full shallow-water reference (SWE) at the Eden peak (the 0.6.0 values are
printed in §1). On the planar embankment it was 1–1.5 m deep at the inflow
where the standard model was 0.3 m deep. Two explanations were possible:

- **measurement** — SGS keeps the water that lies below a cell's mean bed
  (in channels and hollows), which a grid of cell-mean elevations smooths
  away, so its "storage" counts more of the same water; or
- **conveyance** — SGS lets water out of a cell more slowly than it should,
  so the water backs up.

This notebook answers that on a synthetic case built so that every answer is
known in advance: a long, narrow strip on a uniform slope with a straight
channel narrower than a mesh cell (case `sgs_strip`). The answer was
**conveyance**: storage was measured correctly, and the water SGS held was
real water, held back. Two defects held it: overbank, the flow area of a
face stopped growing once the water rose above the cell's highest DEM
sample; in bank, SGS's default limiter capped what a cell could pass by its
volume. Both were fixed in 0.7.0, together with the SGS time step and, on
A5, the geometry of the flow path. The notebook shows

1. the geometry, the meshes and the runs, with their exact commands, and the
   headline numbers (§1–2);
2. the analytic answers — normal depth, storage per metre, arrival times —
   and what they say about measuring storage (§3);
3. how SGS represents the sub-cell channel now, and how it did until 0.6.0,
   by comparing its tables with the terrain directly, without a simulation
   (§4);
4. the runs against the analytic answers and a 2.5 m reference that resolves
   the channel, with the 0.6.0 runs beside them (§5);
5. what was changed and why, with the evidence from this case (§6), what
   remains (§7), and conclusions (§8).

**In short:** SGS now settles at the analytic level of a 2-D model on the
Cartesian 20 m strip, in bank and overbank, within about a centimetre, and
in bank on A5; until 0.6.0 it ran the channel brim-full in bank and piled
the water up metres high overbank, still filling after 8 hours.

**How to run it.** Open it with the kernel **"Julia FloodA5 1.12"** (set-up:
`notebooks/README.md`) and run all cells. The first run makes the case: the
DEM, five meshes, the table diagnostics and 16 simulations, about 40 minutes
with two runs at a time (the 2.5 m SWE reference is the longest) in
`test/case_runs/sgs_strip/` (not tracked by git). Later runs reuse them and
take a minute or two. The 0.6.0 results are not re-run: the old scheme no
longer exists (its meshes are refused). They are read from the record
committed for this purpose, `cases/expected/v0.6.0/`.

**Words used throughout.** *δ*: the water level above the local floodplain,
η − z_f(x) (negative: in bank). *Normal (uniform) flow*: the steady state on
a long uniform slope, the water surface parallel to the bed. *LI*:
local-inertial, `--flow-model standard`. *Cell-mean bed*: the mean DEM
elevation of a cell, which is all a standard mesh knows of its terrain.
*0.6.0*: the SGS scheme up to FloodA5 0.6.0; *now*: the scheme of 0.7.0
(`docs/HYDRAULICS.md` §9).

## 0. Set-up

The cell below finds the repository from the Julia project the kernel was
started with and loads the three modules the notebooks use: `FA`
(`analysis/FloodA5Analysis.jl`, reading outputs), `FC` (`cases/FloodA5Cases.jl`,
the test-case registry; the case is defined in `cases/registry/sgs_strip.jl`)
and `FP` (`visualisation/FloodA5Plots.jl`, the figure style).

`RUN_CASES = false` only analyses runs that already exist. `PARALLEL` runs
that many simulations at once, each with its own thread count (2 for the
20 m and A5 meshes, 4 for the 2.5 m reference: at most 8 threads with
`PARALLEL = 2`).

In [ ]:
# The kernel starts Julia with --project=<repository> (or <repository>/ci):
# walk up from that project to the directory holding FloodModel.jl.
const REPO = let d = dirname(something(Base.active_project(), joinpath(pwd(), "Project.toml")))
    while !isfile(joinpath(d, "FloodModel.jl"))
        dirname(d) == d && error("no FloodA5 repository above $(Base.active_project())")
        d = dirname(d)
    end
    d
end
ENV["FLOODA5_NO_VIS"] = "1"            # the runs need no viewer

include(joinpath(REPO, "analysis", "FloodA5Analysis.jl"));     const FA = FloodA5Analysis
include(joinpath(REPO, "cases", "FloodA5Cases.jl"));           const FC = FloodA5Cases
include(joinpath(REPO, "visualisation", "FloodA5Plots.jl"));   const FP = FloodA5Plots
using CairoMakie, Printf, Statistics
FP.activate!()                         # the FloodA5 figure style

const RUN_CASES = true                 # false: only analyse existing runs
const PARALLEL  = 2                    # simulations at once (each with its own thread count)

println("repository : ", REPO)
println("version    : ", strip(read(joinpath(REPO, "VERSION"), String)))
println("case runs  : ", FC.case_root())

Small helpers: `mdtable` shows a table (as HTML), `f` formats a number,
`readjson` reads a JSON file (the table diagnostics and the 0.6.0 record).

In [ ]:
"An HTML table from a header and rows of already formatted values. (A Markdown table also carries
a LaTeX copy, a `tabular` environment, which VS Code's KaTeX renderer cannot draw.)"
function mdtable(header, rows)
    esc(x) = replace(string(x), "&" => "&amp;", "<" => "&lt;", ">" => "&gt;")
    cell(tag, x) = "<$tag style='text-align:right'>" * esc(x) * "</$tag>"
    s = "<table><thead><tr>" * join(cell("th", h) for h in header) * "</tr></thead><tbody>"
    for r in rows
        s *= "<tr>" * join(cell("td", v) for v in r) * "</tr>"
    end
    return HTML(s * "</tbody></table>")
end
"`x` to `d` decimals, or as 1.2e-05 if it would round to zero."
f(x, d = 3) = !isfinite(x) ? string(x) :
              (x != 0 && abs(x) < 0.5 * 10.0^-d) ? @sprintf("%.1e", x) : string(round(x; digits = d))
"A JSON file as a Dict."
readjson(path) = FA.FO.JSON3.read(read(path, String), Dict{String,Any})
nothing

## 1. The case: a sub-cell channel in a long sloping flume

### Geometry

The terrain (`cases/scripts/sgs_strip_dem.jl`, a 0.5 m DEM) is chosen so that
every answer can be written down without a model:

- a **flume** 4000 m long (x, west → east) and 60 m wide (y), its floodplain a
  plane sloping down the flume at **S = 0.001** (1 m per km);
- a **rectangular channel 5 m wide and 1 m deep** cut along the centre line;
- **no walls**: the floodplain plane continues beyond the strip, and the
  mesh's own edges are closed boundaries except at the downstream end;
- an **outlet reach**: below x = 3800 m the bed steepens to 2.1%. The flow
  passes through critical depth at the break, so whatever each model does at
  its outflow boundary cannot travel back up the flume;
- Manning's **n = 0.03** everywhere;
- a constant inflow (after a 10-minute ramp) into the channel 31 m from the
  upstream end: **3 m³/s**, which stays in the channel (in bank), or
  **20 m³/s**, which spills onto the floodplain (overbank).

The DEM stores the bed at its nodes and the model samples it by bilinear
interpolation, so the model sees exactly this surface except that each
channel wall becomes a ramp 0.5 m wide centred on it. The analytic answers
below use that same surface (`cases/scripts/sgs_strip_geometry.jl`).

### Meshes and runs

| Mesh | Cells | What it sees of the channel |
|---|---|---|
| Cartesian 20 m (`cart20`) | 200 × 3 rows | the channel is a quarter of the middle row; SGS mesh with sub-grid tables, standard mesh with cell-mean beds |
| A5 res 18 (`a5r18`) | pentagons of ~22 m covering the 60 m strip, so its edges are ragged (every cell the strip outline touches: a mean width of 95 m) | as above, but the channel crosses cells at every angle and offset |
| Cartesian 2.5 m (`cart2p5`) | 1600 × 24 | the channel is two cells wide: the **fine reference**, cell-mean beds |

On each mesh the case runs the models at both discharges: **SGS** (the
default, and `--sgs-conveyance lumped`, one hydraulic radius per face,
explained in §4), **local-inertial** (`--flow-model standard`) and **SWE** on
20 m; SGS and local-inertial on A5; local-inertial and SWE on 2.5 m. Coarse
runs last 8 h (a 20 m cell-mean grid fills slowly at 3 m³/s: the water
spreads as a thin sheet over its 60 m width), the 2.5 m runs 4 h; all write a
frame every 5 minutes.

### What is measured

Over the **measuring reach x = 1000–2000 m**, at the last frame:

- **δ**, the water level above the local floodplain, η − z_f(x), in the
  channel cells (and separately in the floodplain cells, if wet). At uniform
  flow it is the same everywhere; in bank δ < 0 (−0.27 m means 0.73 m of
  water in the 1 m channel);
- **storage per metre** of flume (m²): Σ volume / 1000 m, the water a model
  holds; and the **volume above the cell-mean bed**, Σ area·max(η − z̄, 0), what
  a grid of cell-mean elevations would hold at the same water level — the
  two ways of measuring storage in question;
- **steadiness**: the change of the stored volume over the last hour, as a
  fraction of the inflow;
- **arrival**: the time at which the ±50 m window around x = 1000 m and
  2000 m first holds half the 2-D normal storage.

In [ ]:
ss = FC.case("sgs_strip")
FC.show_case(ss)

Reading the commands:

- **prep dem** writes the DEM, the parameters file, the AOI (the flume
  outline, shrunk by 1 cm so the Cartesian generator makes exactly 200 × 3
  cells) and the boundary file `sgs_strip_bc.geojson` (`--bc-file`): a
  *Closed* line round the flume, then a *ZeroGradient* (open) line across the
  downstream end. A feature applies to boundary cells within 1.5 cell
  diameters of it, the later one winning, so the last 1.5 cell diameters of the strip are
  open on every side (the last two columns of the 20 m grid).
- **prep cart20_sgs / a5r18_sgs** build meshes *with SGS tables*
  (`--flow-model sgs --mesh-only`: a storage curve per cell and a table per
  face, §4); **cart20_standard / a5r18_standard / cart2p5_standard** meshes
  with cell-mean beds only. `--cartesian-lat-ref 51` makes the Cartesian
  cells square in the DEM's own metric, so rows run exactly along the
  channel.
- **prep tables_cart20 / tables_a5r18** compare each SGS mesh's tables with
  the terrain, without a simulation (`cases/scripts/sgs_strip_tables.jl`;
  §4).
- **run …** — `--flow-model` is always explicit; `--inflow-point lat,lon,file`
  (latitude first) feeds the hydrograph to the cell containing the point;
  `--bc-file` sets the boundaries; `--dt-max 10` caps the time step (the
  Courant condition sets it in practice); `--sgs-conveyance lumped` selects
  the one-radius face formula (§4). Everything else is the default: for SGS
  that is positivity limiting and a Courant number of 0.5 (§6).

### Run and check

`FC.run_case` makes whatever is missing (first time: the DEM, five meshes,
the table diagnostics and 16 simulations — about 5 minutes of preparation,
a minute per coarse run and 5–25 minutes per 2.5 m run; ~35 minutes with
`PARALLEL = 2`), then computes the metrics and runs the checks. The checks
state what a **correct** model must do. Until 0.6.0 eight of the SGS checks
failed and were kept, unweakened, as expected failures; since 0.7.0 they
pass and are ordinary checks (`CHANGELOG.md` lists the checks that were
changed because they referred to the old scheme).

In [ ]:
t0 = time()
ok_ss = RUN_CASES ? FC.run_case(ss; parallel = PARALLEL) : FC.check(ss)
@printf("\nsgs_strip: %s (%.0f s)\n", ok_ss ? "all checks pass" : "SOME CHECKS FAIL", time() - t0)
m = FC.analyse(ss; io = devnull)        # the metrics (also in test/case_runs/sgs_strip/metrics.toml)
p = FC.sgs_strip_params(FC.case_dir(ss))
a = m["analytic"]
nothing

### The record of 0.6.0, and the headline

`cases/expected/v0.6.0/` holds what the old scheme gave on this case (its
metrics, `sgs_strip_metrics.toml`; its table diagnostics, `tables_*.json`;
the channel-level profile at the last frame and the storage in the measuring
reach through time of its SGS runs, `sgs_strip_series.json`) and on
Carlisle. Its SGS jobs were `cart20_sgs`, `a5r18_sgs` and `cart20_sgspos`
(`--limiter positivity`, a diagnostic run of 0.6.0 used in §6.2); the
analytic answers are the same.

In [ ]:
const V060 = joinpath(REPO, "cases", "expected", "v0.6.0")
_num(x) = x isa AbstractString ? parse(Float64, x) : Float64(x)          # TOML stores NaN as "NaN"
m060 = Dict(j => Dict(k => _num(v) for (k, v) in d) for (j, d) in FC.TOML.parsefile(joinpath(V060, "sgs_strip_metrics.toml")))
S060 = readjson(joinpath(V060, "sgs_strip_series.json"))              # per old SGS job: x, delta_channel, t, store_per_m
carl060 = FC.TOML.parsefile(joinpath(V060, "carlisle_peak_metrics.toml"))
st(j) = carl060[j]["storage_w1_m3"] / 1e6
@printf("Carlisle at the Eden peak (55.5 h, A5 res 18), 0.6.0: SGS stored %.1f Mm³, SWE %.1f, local-inertial %.1f (SGS %.1f× SWE)\n\n",
        st("a5_sgs"), st("a5_swe"), st("a5_li"), st("a5_sgs") / st("a5_swe"))

# Headline: the SGS runs now and in 0.6.0, measuring reach at 8 h
headline = let rows = []
    for (job, ref, what) in (("cart20_sgs_q3", "2d_q3", "Cartesian 20 m, 3 m³/s (in bank)"),
                             ("cart20_sgs_q20", "2d_q20", "Cartesian 20 m, 20 m³/s (overbank)"),
                             ("a5r18_sgs_q3", "2d_q3", "A5 res 18, 3 m³/s (in bank)"),
                             ("a5r18_sgs_q20", "a5w_q20", "A5 res 18, 20 m³/s (overbank)"))
        d, o = m[job], m060[job]
        push!(rows, (what, f(a["delta_$ref"]) * (startswith(ref, "a5w") ? " (2-D, its own width)" : ""),
                     f(d["delta_channel"]), f(d["delta_channel"] - a["delta_$ref"]), f(d["store_per_m"] / a["store_$ref"], 2),
                     f(d["dSdt_rel_end"], 3),
                     f(o["delta_channel"]), f(o["store_per_m"] / a["store_$ref"], 2), f(o["dSdt_rel_end"], 2)))
    end
    mdtable(["SGS run", "analytic δ (m)", "now: δ (m)", "now − analytic (m)", "now: storage ÷ analytic",
             "now: dS/dt ÷ Q", "0.6.0: δ (m)", "0.6.0: storage ÷ analytic", "0.6.0: dS/dt ÷ Q"], rows)
end

(dS/dt ÷ Q is the storage change over the last hour as a fraction of the
inflow: 0 when steady, 1 when the reach keeps everything that flows in.)

## 2. The terrain and the meshes

The cross-section is the same at every x above the outlet reach (relative to
the floodplain level z_f(x) = 10 − 0.001·x m). The figure shows it with the
2-D normal levels of §3 and, below, the three meshes over a 200 m stretch of
the measuring reach with the channel shaded. On A5 the channel zig-zags
through the cells, and many cells hold only a sliver of it (§6.3, §6.4).

In [ ]:
const LABEL = Dict("cart20_sgs" => "SGS, Cartesian 20 m", "cart20_sgslumped" => "SGS (lumped), Cartesian 20 m",
                   "cart20_std" => "LI, Cartesian 20 m", "cart20_swe" => "SWE, Cartesian 20 m",
                   "a5r18_sgs" => "SGS, A5 res 18", "a5r18_std" => "LI, A5 res 18",
                   "fine_std" => "LI, Cartesian 2.5 m", "fine_swe" => "SWE, Cartesian 2.5 m")
"Labels of the 0.6.0 SGS jobs (cases/expected/v0.6.0/)."
const LABEL060 = Dict("cart20_sgs" => "SGS 0.6.0, Cartesian 20 m", "a5r18_sgs" => "SGS 0.6.0, A5 res 18",
                      "cart20_sgspos" => "SGS 0.6.0 (--limiter positivity), Cartesian 20 m")
const QS = (3.0, 20.0); qtag(Q) = "q" * string(round(Int, Q))
jobparts(job) = rsplit(job, '_'; limit = 2)                  # "cart20_sgs_q3" → ["cart20_sgs", "q3"]
label(job; L = LABEL) = L[jobparts(job)[1]] * ", " * replace(jobparts(job)[2], "q" => "") * " m³/s"
runs = Dict(j.name => FA.load_run(FC.job_output(ss, j); label = label(j.name)) for j in ss.jobs)
xy(r) = FA.local_xy(r.mesh, 0.0, 51.0)          # the strip's frame: x along the flume, y across

fig = Figure(size = (900, 820))
ax = Axis(fig[1, 1]; xlabel = "y (m across the flume)", ylabel = "bed − z_f (m)",
          title = "Cross-section of the strip (any x < 3800 m) with the 2-D normal levels")
ys = range(-32, 32; length = 2001)
band!(ax, ys, fill(-1.2, length(ys)), [FC.sgs_strip_zrel(p, y) for y in ys]; color = (FP.INK2, 0.25))
lines!(ax, ys, [FC.sgs_strip_zrel(p, y) for y in ys]; color = FP.INK, linewidth = 1.5, label = "bed (DEM as sampled)")
for (k, Q) in enumerate(QS)
    δ = a["delta_2d_$(qtag(Q))"]
    lines!(ax, [-30, 30], [δ, δ]; color = FP.SERIES[k], linewidth = 2, label = "2-D normal level, $(Int(Q)) m³/s")
end
vlines!(ax, [-10, 10]; color = FP.INK2, linestyle = :dash, linewidth = 0.8)
text!(ax, 0, 1.2; text = "dashed: the 20 m Cartesian rows", align = (:center, :bottom), fontsize = 11, color = FP.INK2)
ylims!(ax, -1.2, 1.6); xlims!(ax, -32, 32); axislegend(ax; position = :lt, framevisible = false, backgroundcolor = (:white, 0.8))

for (k, (job, title)) in enumerate((("cart20_std_q3", "Cartesian 20 m"), ("a5r18_std_q3", "A5 res 18"),
                                     ("fine_std_q3", "Cartesian 2.5 m")))
    r = runs[job]
    axm = Axis(fig[k + 1, 1]; aspect = DataAspect(), title = "$title mesh over x = 1400–1600 m (channel shaded)",
               xlabel = k == 3 ? "x (m along the flume)" : "", ylabel = "y (m)")
    poly!(axm, Rect(1400, -2.5, 200, 5); color = (FP.SERIES[1], 0.35))
    for P in r.mesh.polygons
        x, y = FA.local_xy(P[1, :], P[2, :], 0.0, 51.0)
        (maximum(x) < 1400 || minimum(x) > 1600) && continue
        lines!(axm, x, y; color = FP.INK, linewidth = 0.6)
    end
    xlims!(axm, 1400, 1600); ylims!(axm, -55, 55)
end
fig

## 3. The analytic answers

At **uniform flow** the water surface is parallel to the bed and friction
balances gravity, Q = K(δ)·√S / n, where the conveyance K depends on how the
section is represented. Three representations matter here (all in
`cases/scripts/sgs_strip_geometry.jl`, solved for δ by bisection):

- **2-D** — the depth-averaged equations with no lateral exchange: every strip
  dy of the section carries its own Manning flow, K = ∫ h(y)^(5/3) dy. This is
  what a fine 2-D model converges to (lateral momentum exchange vanishes at
  uniform flow without turbulence closure), so it is the reference for
  everything below — and, since 0.7.0, what the SGS face formula computes
  over each face (§4).
- **cell-mean (`cells20`)** — the three 20 m rows each with its mean bed, as a
  standard 20 m grid sees them: K = Σ 20·(δ − z̄)^(5/3). The middle row's mean
  bed lies 0.25 m below the floodplain (the channel's 5 m² spread over 20 m).
- **lumped (`lumped20`)** — each 20 m row with its *true* section but one
  hydraulic radius, K = Σ A^(5/3)/P_w^(2/3) (A the flow area, P_w the wetted
  width). This is the SGS face formula with one hydraulic radius per face,
  R = A/P (`--sgs-conveyance lumped`; until 0.6.0 the only form), evaluated
  with *exact* face tables.

The last row of each discharge is the 2-D answer for a strip as wide as the
A5 mesh's ragged strip (its mean width over the reach), the fair reference
for A5 overbank.

**Storage per metre** is the water volume per metre of flume at that level;
for the cell-mean grid it is the volume above the cell-mean beds.
**Arrival**: on a dry bed a kinematic wave front (a shock) moves at Q/A_n, A_n
the normal-depth storage per metre, so a station x is reached at
t = 300 s + A_n·(x − 31 m)/Q (the 10-minute ramp delays the volume by 300 s).

In [ ]:
rows = []
for Q in QS, (lab, key) in (("2-D (fine-grid limit)", "2d"), ("cell-mean 20 m rows (standard)", "cells20"),
                            ("lumped 20 m rows (one radius, exact tables)", "lumped20"), ("2-D, the A5 strip's mean width", "a5w"))
    q = qtag(Q)
    push!(rows, (Int(Q), lab, f(a["delta_$(key)_$q"]), f(a["delta_$(key)_$q"] + p.D), f(a["store_$(key)_$q"], 2),
                 f(a["store_$(key)_$q"] / a["store_2d_$q"], 3),
                 round(Int, a["t_arrive_$(key)_$(q)_x1000"]), round(Int, a["t_arrive_$(key)_$(q)_x2000"])))
end
@printf("A5 strip's mean width over the reach: %.1f m\n", a["width_a5"])
mdtable(["Q (m³/s)", "section", "δ (m)", "channel depth (m)", "storage (m²/m)", "÷ 2-D", "arrival x=1000 (s)", "x=2000 (s)"], rows)

### The storage-measurement question

A grid of cell-mean elevations removes the small topographic variations — but
it does not remove their *volume* once they are under water. The cell-mean
bed is the mean of the terrain, so for a water level above the highest point
of a cell, the water above the mean bed, area·(η − z̄), is exactly the water
volume ∫(η − z) dA. Here, above the floodplain (δ ≥ 0), the volume above the
20 m cell-mean beds equals the water volume (the cell-mean row of the table
at 20 m³/s is both), and the 20 m standard grid stores about 8% more than
the 2-D answer at 20 m³/s — because its conveyance is lower (about 4 cm
higher), not because of how it measures storage.

The two measures differ only where a cell is partly dry. In bank (3 m³/s) a
20 m cell-mean grid has no channel, spreads the flow over its whole 20 m row
and then onto the side rows, and stores about 2.2 times the true water volume
(the table). So *at low flow* SGS should store **less** than a standard grid,
and at high flow about the same. A model that stores 2.5 times as much as
SWE at a flood peak is not explained by the storage measure: it is holding
water back. Until 0.6.0 SGS did exactly that on this strip (§5); §4 and §6
say why, and what was changed.

## 4. How SGS represents the sub-cell channel

An SGS mesh carries tables built from the DEM (`build_sgs_tables!`,
`mesh/A5Grid.jl`; `docs/HYDRAULICS.md` §9.1):

- **for every cell, a storage curve**: the stored volume V(η) and the wetted
  area at 100 levels from the cell's lowest to its highest ground, sampled on
  a regular working grid over the cell (`--sgs-samples`, default 1600
  points). The model inverts it to get η from the volume
  (`wse_from_volume`); above the highest ground the level rises with the
  full cell area. Storage is a level pool.
- **for every face, one conveyance table**, held identically by both cells
  (`face_table_from_profile`, `face_apk`). From the DEM profile z_p along the
  face (points of width dx, half a DEM pixel apart):
  flow area A(η) = Σ dx·(η − z_p)⁺, wetted width P(η) = Σ dx·[η > z_p] and the
  conveyance integral K(η) = Σ dx·((η − z_p)⁺)^(5/3), at 64 levels from the
  face's sill to above its highest point. Above the top level every point is
  wet: A grows with the full face width W, P = W and K = W·(d + c)^(5/3)
  (d the height above the top level, c fixed by K there). Nothing stops
  growing with stage. (Each point conveys only from the level at which it is
  connected to the water of both cells, which closes barriers inside a cell:
  `docs/HYDRAULICS.md` §9.3, case `planar_wall`. The strip has no barriers.)

The flux kernel (`_face_flux_sgs!`, `surfacewater/kernels.jl`) evaluates the
face at the higher of the two cells' levels and uses Manning's law in its
R–A form with a hydraulic radius from `_sgs_radius`
(`surfacewater/flow2d.jl`): by default R = (K/A)^(3/2), so A·R^(2/3) = K, the
2-D conveyance of the face — every strip of it carries its own Manning flow;
with `--sgs-conveyance lumped`, R = A/P, one radius for the whole face.

**Until 0.6.0** a face had no table of its own. Each *cell* tabulated the
flow area and wetted width of each of its faces at *its own* 100 storage
levels, from its z_min to its z_max, and the lookup returned the top value
for any level above z_max. The kernel took the average of the two cells'
values, A = ½(A_i + A_j) and R = ½(R_i + R_j). On a gently sloping floodplain
a cell's z_max is a few centimetres above its lowest point (on this strip a
20 m cell spans 2 cm of floodplain), so **the flow area of a face stopped
growing just above bankfull**, and a face point above one cell's z_max was
never wet on that side, halving both averages.

The figure compares, for the Cartesian 20 m cell in the middle of the strip
(in the channel row), the flow area and the conveyance of two of its faces
with the terrain, integrated directly (`cases/scripts/sgs_strip_tables.jl`;
levels relative to the floodplain at the cell centre). The 0.6.0 conveyance
is the kernel's A·R^(2/3) from the averaged values.

In [ ]:
T    = Dict(mesh => FC.sgs_strip_tables(FC.case_dir(ss), mesh) for mesh in ("cart20", "a5r18"))   # now
T060 = Dict(mesh => readjson(joinpath(V060, "tables_$mesh.json")) for mesh in ("cart20", "a5r18"))  # 0.6.0
c, c0 = T["cart20"]["cell"], T060["cart20"]["cell"]
@assert c["x"] ≈ c0["x"] && c["y"] ≈ c0["y"]      # the same cell in both records
zf = c["z_floodplain"]
F(v) = Float64.(v)
lev(cc) = F(cc["wse"]) .- zf                     # each record has its own level grid
fig = Figure(size = (950, 800))
for (k, (key, ttl)) in enumerate((("face_downstream", "downstream face (across the channel)"),
                                   ("face_lateral", "lateral face (floodplain)")))
    e, e0 = c[key], c0[key]
    axA = Axis(fig[1, k]; xlabel = "face flow area A (m²)", ylabel = "water level − z_f (m)", title = "Flow area, $ttl")
    axK = Axis(fig[2, k]; xlabel = "face conveyance A·R^(2/3) (m^(8/3))", ylabel = "water level − z_f (m)",
               title = "Conveyance, $ttl")
    lines!(axA, F(e["A_analytic"]), lev(c); color = FP.INK, linewidth = 4, label = "terrain (exact)")
    lines!(axA, F(e0["A_edge"]), lev(c0); color = FP.SERIES[2], linewidth = 2.5, label = "0.6.0: ½(A_i + A_j)")
    lines!(axA, F(e["A_edge"]), lev(c); color = FP.SERIES[1], linewidth = 2, linestyle = :dash, label = "now: the face's table")
    lines!(axK, F(e["K_analytic"]), lev(c); color = FP.INK, linewidth = 4, label = "terrain: Σ dx·h^(5/3)")
    lines!(axK, F(e0["A_edge"]) .* F(e0["R_edge"]) .^ (2 / 3), lev(c0); color = FP.SERIES[2], linewidth = 2.5,
           label = "0.6.0: ½(A_i + A_j)·(½(R_i + R_j))^(2/3)")
    lines!(axK, F(e["K_edge"]), lev(c); color = FP.SERIES[1], linewidth = 2, linestyle = :dash, label = "now: K of the face's table")
    for ax in (axA, axK)
        hlines!(ax, [c0["z_max"] - zf]; color = FP.SERIES[2], linestyle = :dot, linewidth = 1)
        hlines!(ax, [e["top_table"] - zf]; color = FP.SERIES[1], linestyle = :dot, linewidth = 1)
        ylims!(ax, -1.05, 1.5); xlims!(ax, 0, nothing)
    end
    if k == 1
        axislegend(axA; position = :rb, framevisible = false)
        axislegend(axK; position = :rb, framevisible = false)
    end
end
Label(fig[3, 1:2], "dotted: orange, the cell's highest DEM sample (0.6.0: face flow area frozen above it); " *
      "blue, the top level of the face's table (now: continued analytically above it; for the downstream face above the plotted range)";
      fontsize = 11, color = FP.INK2, tellwidth = false)
@printf("Cell at x = %.0f m: z_max − z_f = %+.3f m.  Downstream face: sill − z_f = %+.3f m, table top − z_f = %+.3f m; lateral face: %+.3f, %+.3f m\n",
        c["x"], c0["z_max"] - zf, c["face_downstream"]["sill_table"] - zf, c["face_downstream"]["top_table"] - zf,
        c["face_lateral"]["sill_table"] - zf, c["face_lateral"]["top_table"] - zf)
fig

Over all cells and faces of the measuring reach, the table errors at levels
from 0.75 m below to 1 m above the floodplain, now and (in brackets) in
0.6.0. Stored volume: the error relative to the larger of V and 5 cm over the
cell (median and largest |·| over the cells). Faces: the median ratio of the
flow area the kernel uses to the face's true flow area, and the same for the
conveyance K (0.6.0's record has no conveyance ratio). In bank the floodplain
faces carry no water ("—"; the few A5 faces that only clip the channel's
0.5 m wall ramp are left out).

In [ ]:
g(d, k) = Float64(something(get(d, k, NaN), NaN))
rows = []
for mesh in ("cart20", "a5r18"), δ in T[mesh]["levels"]
    t, t0 = T[mesh], T060[mesh]; s = string(Float64(δ))
    both(x, x0, d = 3) = f(x, d) * " (" * f(x0, d) * ")"
    fp(tab, key) = δ < 0 ? NaN : g(tab[key]["floodplain"][s], "median")
    push!(rows, (mesh == "cart20" ? "Cartesian 20 m" : "A5 res 18", f(δ, 2),
                 f(g(t["volume_rel_err"][s], "median"), 4), both(g(t["volume_rel_err"][s], "max_abs"), g(t0["volume_rel_err"][s], "max_abs")),
                 both(g(t["edge_area_ratio"]["channel"][s], "median"), g(t0["edge_area_ratio"]["channel"][s], "median")),
                 δ < 0 ? "—" : both(fp(t, "edge_area_ratio"), fp(t0, "edge_area_ratio")),
                 f(g(t["edge_conveyance_ratio"]["channel"][s], "median")),
                 δ < 0 ? "—" : f(fp(t, "edge_conveyance_ratio"))))
end
mdtable(["mesh", "δ (m)", "V error, median", "V error, max |·| (0.6.0)", "A ratio, channel faces (0.6.0)",
         "A ratio, floodplain faces (0.6.0)", "K ratio, channel faces", "K ratio, floodplain faces"], rows)

What this means for conveyance: the discharge one cross-section of the
Cartesian 20 m strip can carry at uniform flow on the bed slope, from the
tables as the kernel uses them (three faces, each at the upstream cell's
level), against the analytic conveyances of §3. Now the tables give both
forms: the default (2-D, K·√S/n) and the lumped one (A·(A/P)^(2/3)·√S/n).

In [ ]:
u, u0 = T["cart20"]["uniform_flow"], T060["cart20"]["uniform_flow"]
δs = range(-0.95, 3.0; length = 300)
Qof(mode; kw...) = [FC.sgs_strip_conveyance(p, δ, mode; kw...) * sqrt(p.S) / FC.SGS_STRIP_N for δ in δs]
fig = Figure(size = (850, 470))
ax = Axis(fig[1, 1]; xlabel = "uniform-flow discharge (m³/s)", ylabel = "level above the floodplain δ (m)",
          title = @sprintf("Discharge a cross-section carries at the bed slope, Cartesian 20 m (faces at x = %.0f m)", u["x_face"]))
lines!(ax, Qof(:twod), δs; color = FP.INK, linewidth = 4, label = "analytic 2-D (fine-grid limit)")
lines!(ax, Qof(:lumped; rows = 20.0), δs; color = FP.SERIES[3], linewidth = 4, label = "analytic lumped 20 m rows")
lines!(ax, Qof(:cells; rows = 20.0), δs; color = FP.MUTED, linewidth = 1.5, linestyle = :dash, label = "analytic cell-mean 20 m rows (standard)")
lines!(ax, F(u["Q"]), F(u["delta"]); color = FP.SERIES[1], linewidth = 2, linestyle = :dash, label = "SGS tables now, 2-D form (default)")
lines!(ax, F(u["Q_lumped"]), F(u["delta"]); color = FP.SERIES[4], linewidth = 2, linestyle = :dash, label = "SGS tables now, lumped form")
lines!(ax, F(u0["Q"]), F(u0["delta"]); color = FP.SERIES[2], linewidth = 2.5, label = "SGS tables 0.6.0")
vlines!(ax, [3.0, 20.0]; color = FP.MUTED, linestyle = :dot)
xlims!(ax, 0, 40); ylims!(ax, -1, 1.2)
axislegend(ax; position = :rb, framevisible = false)
"δ at which a discharge curve (Q against δ, increasing) reaches Q0, by linear interpolation."
function δ_at(Qv, δv, Q0)
    k = findfirst(>=(Q0), Qv); k === nothing && return NaN; k == 1 && return δv[1]
    return δv[k-1] + (Q0 - Qv[k-1]) / (Qv[k] - Qv[k-1]) * (δv[k] - δv[k-1])
end
@printf("0.6.0 tables: at most %.2f m³/s at the bed slope (reached at δ = %.2f m); 20 m³/s needed a water-surface slope of %.4f (%.0f× the bed)\n",
        u0["Q_max"], u0["delta_at_Q_max"], p.S * (20 / u0["Q_max"])^2, (20 / u0["Q_max"])^2)
for Q0 in QS
    q = qtag(Q0)
    @printf("now, %2d m³/s at the bed slope: tables 2-D form δ = %+.3f m (analytic 2-D %+.3f); lumped form δ = %+.3f m (analytic lumped %+.3f)\n",
            Q0, δ_at(F(u["Q"]), F(u["delta"]), Q0), a["delta_2d_$q"], δ_at(F(u["Q_lumped"]), F(u["delta"]), Q0), a["delta_lumped20_$q"])
end
fig

## 5. Results

### Steady state in the measuring reach

Per run, at its last frame (8 h; 2.5 m runs 4 h), over x = 1000–2000 m: the
level above the floodplain in the channel cells and in the floodplain cells
(if wet), their spread over all wet cells, the storage per metre (water
volume, and the volume above the cell-mean bed), the storage relative to the
2-D answer (for A5: the 2-D answer for its own mean width), the storage
change over the last hour as a fraction of the inflow (0 when steady), the
arrival times and the mass-balance error. The last rows of each discharge
are the 0.6.0 SGS runs, from the record. The analytic values to compare with
are in the table of §3.

In [ ]:
order = ["fine_std", "fine_swe", "cart20_std", "cart20_swe", "cart20_sgs", "cart20_sgslumped", "a5r18_std", "a5r18_sgs"]
order060 = ["cart20_sgs", "cart20_sgspos", "a5r18_sgs"]
rows = []
dry(x) = isnan(x) ? "dry" : f(x)
tfmt(t) = isfinite(t) ? round(Int, t) : "never"
function steady_row(lab, d, ref)
    (lab, f(d["delta_channel"]), dry(d["delta_floodplain"]), f(d["delta_spread"]),
     f(d["store_per_m"], 2), f(d["store_above_mean_per_m"], 2), f(d["store_per_m"] / a["store_$ref"], 2),
     f(d["dSdt_rel_end"], 3), tfmt(d["t_arrive_x1000"]), tfmt(d["t_arrive_x2000"]), f(d["mass_balance_rel"], 12))
end
for Q in QS
    q = qtag(Q)
    refof(o) = startswith(o, "a5") ? "a5w_$q" : "2d_$q"
    for o in order
        push!(rows, steady_row(label(o * "_" * q), m[o * "_" * q], refof(o)))
    end
    push!(rows, ("analytic 2-D, $(Int(Q)) m³/s", f(a["delta_2d_$q"]), Q > 5 ? f(a["delta_2d_$q"]) : "dry", "0",
                 f(a["store_2d_$q"], 2), "", "1.0", "0", round(Int, a["t_arrive_2d_$(q)_x1000"]), round(Int, a["t_arrive_2d_$(q)_x2000"]), ""))
    for o in order060
        push!(rows, steady_row(label(o * "_" * q; L = LABEL060), m060[o * "_" * q], refof(o)))
    end
end
mdtable(["run", "δ channel (m)", "δ floodplain (m)", "spread (m)", "storage (m²/m)", "above mean bed (m²/m)",
         "÷ 2-D (A5: own width)", "dS/dt ÷ Q", "arrival 1000 m (s)", "2000 m (s)", "mass error"], rows)

### Long profiles

The level above the floodplain along the flume at the last frame, in the
channel cells (mean per 20 m of x): uniform flow is a horizontal line. The
outlet reach starts at x = 3800 m; the measuring reach is shaded, the
channel (δ < 0) greyed. Dotted coloured lines: the 0.6.0 SGS runs (from the
record); black dotted: the analytic 2-D level. The second panel is a
close-up of the first around the in-bank levels; the last shows the 0.6.0
overbank runs on their own scale.

In [ ]:
"Mean δ of the wet channel cells per bin of x (m)."
function chan_profile(r; bin = 20.0)
    pr = FC.sgs_strip_profile(r, FA.nframes(r), p)
    edges = 0:bin:3800
    xs = Float64[]; δs = Float64[]
    for x0 in edges[1:end-1]
        sel = [i for i in eachindex(pr.x) if pr.chan[i] && pr.wet[i] && x0 <= pr.x[i] < x0 + bin]
        isempty(sel) && continue
        push!(xs, x0 + bin / 2); push!(δs, mean(pr.δ[sel]))
    end
    return xs, δs
end
"The 0.6.0 channel profile of an old SGS job (20 m bins; null where no channel cell was wet)."
old_profile(job) = (F(S060[job]["x"]), [v === nothing ? NaN : Float64(v) for v in S060[job]["delta_channel"]])
style = Dict("fine_std" => (FP.INK, :solid, 3), "fine_swe" => (FP.INK2, :dash, 2),
             "cart20_std" => (FP.SERIES[2], :solid, 1.5), "cart20_swe" => (FP.SERIES[2], :dash, 1.5),
             "cart20_sgs" => (FP.SERIES[1], :solid, 2.5), "cart20_sgslumped" => (FP.SERIES[1], :dash, 2),
             "a5r18_std" => (FP.SERIES[3], :solid, 1.5), "a5r18_sgs" => (FP.SERIES[4], :solid, 2.5))
style060 = Dict("cart20_sgs" => (FP.SERIES[1], :dot, 2.5), "cart20_sgspos" => (FP.SERIES[5], :dot, 2.5),
                "a5r18_sgs" => (FP.SERIES[4], :dot, 2.5))
fig = Figure(size = (950, 1750))
panels = ((3.0, (-0.35, 0.25), "3 m³/s (in bank)", true),
          (3.0, (-0.30, -0.24), "3 m³/s (in bank), close-up", true),
          (20.0, (0.2, 0.6), "20 m³/s (overbank), the runs now", false))
for (k, (Q, yl, ttl, with060)) in enumerate(panels)
    q = qtag(Q)
    ax = Axis(fig[k, 1]; xlabel = "x (m along the flume)", ylabel = "δ = η − z_f (m)",
              title = "Water level above the floodplain in the channel cells at the last frame, $ttl")
    hspan!(ax, [-1.0], [0.0]; color = (FP.MUTED, 0.12))
    vspan!(ax, [FC.SGS_STRIP_REACH[1]], [FC.SGS_STRIP_REACH[2]]; color = (FP.SERIES[3], 0.07))
    for o in order
        c_, ls, lw = style[o]; x, d = chan_profile(runs[o * "_" * q])
        lines!(ax, x, d; color = c_, linestyle = ls, linewidth = lw, label = LABEL[o])
    end
    if with060
        for o in order060
            c_, ls, lw = style060[o]; x, d = old_profile(o * "_" * q)
            lines!(ax, x, d; color = c_, linestyle = ls, linewidth = lw, label = LABEL060[o])
        end
    end
    if k > 1         # the analytic levels, in a legend of their own
        refs = [hlines!(ax, [a["delta_2d_$q"]]; color = FP.INK, linestyle = :dot, linewidth = 1.5),
                hlines!(ax, [a["delta_lumped20_$q"]]; color = FP.SERIES[3], linestyle = :dot, linewidth = 1.5)]
        refnames = ["analytic 2-D", "analytic lumped 20 m rows"]
        if Q > 5
            push!(refs, hlines!(ax, [a["delta_a5w_$q"]]; color = FP.SERIES[4], linestyle = :dot, linewidth = 1.5))
            push!(refnames, "analytic 2-D, the A5 strip's width")
        end
        axislegend(ax, refs, refnames; position = Q > 5 ? :lb : :lt, framevisible = true, backgroundcolor = (:white, 0.85),
                   labelsize = 10)
    else
        hlines!(ax, [a["delta_2d_$q"]]; color = FP.INK, linestyle = :dot, linewidth = 1)
    end
    ylims!(ax, yl...); xlims!(ax, 0, 3800)
end
Legend(fig[4, 1], contents(fig[1, 1])[1]; orientation = :horizontal, nbanks = 4, framevisible = false, labelsize = 11)
ax = Axis(fig[5, 1]; xlabel = "x (m along the flume)", ylabel = "δ (m)", yscale = Makie.pseudolog10,
          yticks = [0, 0.2, 0.5, 1, 2, 5, 10, 20], title = "20 m³/s: the SGS runs of 0.6.0 (dotted) and now (solid), on a log scale")
vspan!(ax, [FC.SGS_STRIP_REACH[1]], [FC.SGS_STRIP_REACH[2]]; color = (FP.SERIES[3], 0.07))
for o in ("cart20_sgs", "a5r18_sgs")
    c_, ls, lw = style[o]; x, d = chan_profile(runs[o * "_q20"])
    lines!(ax, x, d; color = c_, linestyle = ls, linewidth = lw, label = LABEL[o])
end
for o in order060
    c_, ls, lw = style060[o]; x, d = old_profile(o * "_q20")
    lines!(ax, x, d; color = c_, linestyle = ls, linewidth = lw, label = LABEL060[o])
end
c_, ls, lw = style["fine_std"]; x, d = chan_profile(runs["fine_std_q20"])
lines!(ax, x, d; color = c_, linestyle = ls, linewidth = lw, label = LABEL["fine_std"])
ylims!(ax, 0, 25); xlims!(ax, 0, 3800); axislegend(ax; position = :rt, framevisible = false, labelsize = 10)
fig

### Filling

Storage per metre in the measuring reach against time. A correct model fills
the reach as the kinematic front passes (§3 arrival times) and then holds the
normal-depth storage (black dotted lines: the 2-D answer). Dotted coloured
lines: the 0.6.0 SGS runs.

In [ ]:
function reach_storage(r)
    x, _ = xy(r); sel = [i for i in eachindex(x) if FC.SGS_STRIP_REACH[1] <= x[i] < FC.SGS_STRIP_REACH[2]]
    L = FC.SGS_STRIP_REACH[2] - FC.SGS_STRIP_REACH[1]
    return r.t ./ 3600, [sum(v -> isfinite(v) ? v : 0.0, FA.volume(r, k)[sel]) / L for k in 1:FA.nframes(r)]
end
fig = Figure(size = (950, 560))
for (k, Q) in enumerate(QS)
    q = qtag(Q)
    ax = Axis(fig[1, k]; xlabel = "time (h)", ylabel = "storage, x = 1000–2000 m (m²/m)",
              title = "Storage in the measuring reach, $(Int(Q)) m³/s", yscale = Q > 5 ? log10 : identity)
    for o in order
        c_, ls, lw = style[o]; t, s = reach_storage(runs[o * "_" * q])
        Q > 5 && (s = max.(s, 1.0))
        lines!(ax, t, s; color = c_, linestyle = ls, linewidth = lw, label = LABEL[o])
    end
    for o in order060
        c_, ls, lw = style060[o]; s0 = S060[o * "_" * q]
        s = F(s0["store_per_m"]); Q > 5 && (s = max.(s, 1.0))
        lines!(ax, F(s0["t"]) ./ 3600, s; color = c_, linestyle = ls, linewidth = lw, label = LABEL060[o])
    end
    hlines!(ax, [a["store_2d_$q"]]; color = FP.INK, linestyle = :dot)
    k == 1 && ylims!(ax, 0, 20)
    if k == 2
        tk = [1, 2, 5, 10, 20, 50, 100, 200]
        ax.yticks = (tk, string.(tk)); ylims!(ax, 1, 300)
    end
end
Legend(fig[2, 1:2], contents(fig[1, 1])[1]; orientation = :horizontal, nbanks = 4, framevisible = false, labelsize = 11)
fig

## 6. What was changed and why

The strip separates three candidate causes — the storage accounting, the
tables, the flux calculation — because each can be checked on its own.

**Storage accounting was not the cause.** The SGS storage tables reproduced
the stored volume of the terrain closely in 0.6.0 as now (§4 table: well
under 1% on the 20 m mesh; on A5 a median near zero, and a largest error per
level of a few per cent now, more in 0.6.0). Overbank, SGS's water volume
and its volume above the cell-mean bed agree to within 1% (the
*measurement* check of the case), as they must. SGS stored more because its
water level was higher, not because it counted water a standard grid cannot
hold. Four things were changed (`CHANGELOG.md`, `docs/HYDRAULICS.md` §9);
the sections below take them in turn with the evidence from this case.

### 6.1 Face tables and the 2-D conveyance

**Defect (overbank).** The 0.6.0 face flow areas stopped at the cell's
highest DEM sample and were averaged with the neighbour's (§4): at
0.4 m above the floodplain the 20 m faces carried a fraction of their true
flow area, and one cross-section could carry only about 5 m³/s at the bed
slope. To pass 20 m³/s the water surface had to steepen many times over, so
the level rose towards the inflow without limit and the flume kept filling —
the dotted overbank curves of §5.

**Change.** One table per face, from the face's own DEM profile on the
face's own levels, continued above its highest point, shared by both cells
(`face_table_from_profile`, `face_apk`); the kernel evaluates it once at the
higher of the two levels. The §4 figures and table show it matching the
terrain at every level.

**The conveyance form.** With exact tables, one hydraulic radius per face
(R = A/P, the 0.6.0 form) gives the level of a cell-mean grid overbank, not
the 2-D level: a face that is half channel and half floodplain has one mean
depth, and Manning's law is not linear in depth. The default is now
R = (K/A)^(3/2), so the kernel's A·R^(2/3) is K = Σ dx·h^(5/3), the 2-D
conveyance of the face. Both forms run on this case: the lumped runs settle
at the lumped analytic level, the default runs at the 2-D level.

In [ ]:
for q in ("q3", "q20")
    for (job, ref) in (("cart20_sgs_$q", "2d_$q"), ("cart20_sgslumped_$q", "lumped20_$q"))
        d = m[job]
        @printf("%-42s δ %+.3f m against analytic %-8s %+.3f m (%+.3f);  storage %5.2f m²/m against %5.2f\n",
                label(job), d["delta_channel"], ref, a["delta_$ref"], d["delta_channel"] - a["delta_$ref"],
                d["store_per_m"], a["store_$ref"])
    end
end
@printf("\nlumped − 2-D, overbank: analytic %+.3f m; runs %+.3f m\n",
        a["delta_lumped20_q20"] - a["delta_2d_q20"], m["cart20_sgslumped_q20"]["delta_channel"] - m["cart20_sgs_q20"]["delta_channel"])

### 6.2 The limiter

**Defect (in bank).** SGS ran with the *legacy donor cap* by default
(`--scheme recommended` gave SGS `--limiter legacy`). It allows an edge to
take at most V/(2N) of the donor cell's volume in one step (N = 4 faces on a
Cartesian grid, 5 on A5). A channel cell that must pass Q downstream through
one face can then do so only when it holds V ≥ 2N·Q·Δt, however good its
tables. At 3 m³/s with the 0.6.0 step of about 4.3 s on the 20 m mesh that
is about 100 m³ — the brim-full channel cell, where the 2-D normal depth
needs about 71 m³ (fix plan, §2.1 S3: 102 m³ predicted, 100.6 m³ observed).
The 0.6.0 SGS run settled there, with the channel at the bank top; the
0.6.0 diagnostic run with `--limiter positivity` and the same (old) tables
settled at the lumped analytic level instead:

In [ ]:
Vbrim, Vn = 20 * FC.sgs_strip_storage(p, 0.0), 20 * FC.sgs_strip_storage(p, a["delta_2d_q3"])
@printf("a 20 m channel cell holds %.1f m³ brim-full (δ = 0) and %.1f m³ at the 2-D normal level\n", Vbrim, Vn)
@printf("2N·Q·Δt with N = 4, Q = 3 m³/s and the 0.6.0 step of about 4.3 s (fix plan §2.2 C1): %.0f m³\n\n", 8 * 3.0 * 4.3)
for (job, ref) in (("cart20_sgs_q3", "2d_q3"), ("cart20_sgspos_q3", "lumped20_q3"))
    d = m060[job]
    @printf("%-58s δ %+.3f m (analytic %s %+.3f);  storage %.2f m²/m = %.0f m³ per 20 m of flume\n",
            label(job; L = LABEL060), d["delta_channel"], ref, a["delta_$ref"], d["store_per_m"], 20 * d["store_per_m"])
end
r = runs["cart20_sgs_q3"]; dt_now = r.t[end] / r.info["summary"]["n_steps"]
@printf("\nnow (positivity): %s δ %+.3f m.  Its mean step is %.2f s, so the legacy cap would need 2N·Q·Δt = %.0f m³ — still more than %.0f m³\n",
        label("cart20_sgs_q3"), m["cart20_sgs_q3"]["delta_channel"], dt_now, 8 * 3.0 * dt_now, Vn)

**Change.** Positivity limiting for SGS, in the interior and at open
boundaries (`--scheme recommended` gives SGS `--limiter positivity` and
`--boundary-limiter positivity`): one scale factor per donor cell, applied
only when its total outflow in a step would exceed its volume. It does not
bind on a steadily flowing channel. As the last line shows, the new time
step alone would not have removed the cap: it binds whenever a cell's
discharge times the time step is a sizeable fraction of its volume — deep,
fast channel flow through few faces — and it scales with Δt, so with cell
size. (`docs/HYDRAULICS.md` §15.7 found the SGS donor cap binding on half of
all wet edge-steps at Carlisle.) `--limiter legacy` still selects it.

### 6.3 The time step

**Defect.** The SGS time step used the depth of each cell capped at its
relief, z_max − z_min. A floodplain cell of this strip has 2 cm of relief,
so its wave speed was taken as √(g·0.02) ≈ 0.44 m/s whatever the depth; the
step was set by the channel cells' 1 m of relief, however deep the water
above. The frozen conveyance and the donor cap kept this stable; with correct
conveyance it would not have been (fix plan §2.2, C1).

**Change** (`_cfl_dt_local`; `docs/HYDRAULICS.md` §9.4). Two limits:

1. the **hydraulic depth** V/A_wet of the cell (the mean depth of its wet
   part) in the usual ℓ/√(g h) form (`_sgs_cfl_depth`);
2. a **face-area limit** (`_sgs_face_rate`): a level change of a cell changes
   its net inflow in proportion to Σ A_f/L_f over its faces, so its level
   oscillates at ω = √(g·Σ(A_f/L_f)/A_wet) and Δt ≤ 2·courant/ω. For a fully
   wet square cell this is limit 1; for a cell that holds only a sliver of a
   channel that its faces convey, it is shorter by √(cell length / sliver
   length). Without it such cells oscillated and held the channel up, by up
   to 0.5 m between neighbouring cells on the A5 strip (`CHANGELOG.md`).

The SGS Courant number is also 0.5 by default (0.7 for the other models;
§7). On this strip, how much of each mesh's channel is held in slivers, and
what the steps cost:

In [ ]:
"Polygon area (shoelace)."
parea(x, y) = (n = length(x); abs(sum(x[i] * y[mod1(i + 1, n)] - x[mod1(i + 1, n)] * y[i] for i in 1:n)) / 2)
"Clip a polygon to the half-plane s·(y − y0) ≤ 0 (Sutherland–Hodgman)."
function clip_y(x, y, y0, s)
    X = Float64[]; Y = Float64[]; n = length(x)
    for i in 1:n
        j = mod1(i + 1, n)
        ina, inb = s * (y[i] - y0) <= 0, s * (y[j] - y0) <= 0
        ina && (push!(X, x[i]); push!(Y, y[i]))
        ina != inb && (push!(X, x[i] + (y0 - y[i]) / (y[j] - y[i]) * (x[j] - x[i])); push!(Y, y0))
    end
    return X, Y
end
"Share of a cell's area in the channel (|y − YC| ≤ W/2), from its polygon."
function chan_share(P)
    x, y = FA.local_xy(P[1, :], P[2, :], 0.0, 51.0)
    cx, cy = clip_y(x, y, p.YC + p.W / 2, 1); cx, cy = clip_y(cx, cy, p.YC - p.W / 2, -1)
    return length(cx) < 3 ? 0.0 : parea(cx, cy) / parea(x, y)
end
for (mesh, job) in (("Cartesian 20 m", "cart20_sgs_q3"), ("A5 res 18", "a5r18_sgs_q3"))
    r = runs[job]; x, _ = xy(r)
    sh = [chan_share(r.mesh.polygons[i]) for i in eachindex(x) if FC.SGS_STRIP_REACH[1] <= x[i] < FC.SGS_STRIP_REACH[2]]
    sh = filter(>(0), sh)
    @printf("%-15s %3d cells of the reach hold part of the channel: share of the cell %.3f–%.3f (median %.3f); %d hold less than 5%%\n",
            mesh, length(sh), minimum(sh), maximum(sh), median(sh), count(<(0.05), sh))
end
@printf("\nspread of δ over the wet cells of the reach in bank now: Cartesian %.1e m, A5 %.3f m\n\n",
        m["cart20_sgs_q3"]["delta_spread"], m["a5r18_sgs_q3"]["delta_spread"])
rows = []
for q in ("q3", "q20"), o in ("cart20_sgs", "cart20_sgslumped", "cart20_std", "cart20_swe", "a5r18_sgs", "a5r18_std")
    r = runs[o * "_" * q]; s = r.info["summary"]
    push!(rows, (label(o * "_" * q), r.info["options"]["courant"], s["n_steps"], f(r.t[end] / s["n_steps"], 2)))
end
mdtable(["run (8 h)", "Courant", "steps", "mean Δt (s)"], rows)

The 0.6.0 record has no step counts. The fix plan quotes them for A5 in
bank: 5 650 steps in 0.6.0 against about 24 300 now (§9.2); the table shows
the step count now. On the Cartesian mesh, where every channel cell holds
the channel along its whole length, SGS takes 1.5–2 times the steps of
local-inertial (a factor 0.7/0.5 = 1.4 is the Courant number); on A5, where
many cells hold a sliver, several times as many.

### 6.4 Flow-path geometry on A5

With the face tables, positivity and the new time step, the Cartesian runs
reached the analytic level, but A5 in bank stayed **0.054 m high** (fix plan
§9; `docs/HYDRAULICS.md` §9.2). On A5 the channel zig-zags through the cells
(§2): the kernel took the water-surface slope over the distance between cell
centres, and the face's full flow area, although the water of a partly wet
cell lies around its wetted area, on a line shorter than the line through
the centres, and the faces cut that line obliquely, wider than the channel
is.

**Change** (`--sgs-path on`, the default; `_sgs_path`, `wet_centroid`). For
cells less than half wet the slope is taken over the distance between the
two cells' wetted-area centroids and the face's flow area is projected on
that line; fully wet cells keep the kernel's own geometry, with a blend in
between. The 0.054 m was measured during the work and is not in the 0.6.0
record (the 0.6.0 A5 run was out of bank); the result now:

In [ ]:
for q in ("q3", "q20")
    job = "a5r18_sgs_$q"; ref = q == "q3" ? "2d_q3" : "a5w_q20"; r = runs[job]
    @printf("%-28s --sgs-path %-5s δ %+.3f m against analytic %-8s %+.3f m (%+.3f m)\n", label(job),
            r.info["options"]["sgs_path"] ? "on" : "off", m[job]["delta_channel"], ref, a["delta_$ref"],
            m[job]["delta_channel"] - a["delta_$ref"])
end

## 7. Limits and what remains

- **About 1 cm low.** Every SGS run on the 20 m mesh, and A5 in bank, settles
  1–2 cm below its analytic level (the 2-D level for the default runs, the
  lumped level for the lumped runs; §6.1, §6.4, and the cell below). The fix
  plan (§9.2) attributes it to evaluating the face at the upstream cell's
  level, which on this slope lies about 1 cm above the level at the face
  (first order in cell size). The discharge curves of the tables in §4,
  evaluated that way, support it: on the Cartesian mesh they give, at the
  bed slope, the levels the runs settle at, in both forms. It is inside the
  case's 3 cm tolerance.
- **A5 overbank.** The A5 strip is ragged (every cell the strip outline
  touches), so its edges hold slack water, and no analytic answer is exact
  for it. SGS sits between the 2-D answer for its own mean width and
  local-inertial on the same mesh (§5; the check asks that it be no further
  from that answer than local-inertial is).
- **Cost.** SGS takes several times the steps of local-inertial on A5 (§6.3),
  through the face-area limit for cells holding a sliver of a channel and
  the lower Courant number.
- **Courant 0.5.** At 0.7 the SGS water surface in deep, fast channel flow
  broke into cell-scale oscillations on Carlisle near the peak; 0.5 and 0.35
  agree (fix plan §9.3). Standard flow damps such oscillations with its η
  filter, which SGS does not have yet; with it SGS could return to 0.7.
- **Carlisle** (`notebooks/03_carlisle.ipynb`, case `carlisle_peak`). The
  2.5× storage is gone: SGS now stores less than local-inertial on the same
  mesh (fix plan §9.3). Against 5 m Cartesian references (case
  `carlisle_peak_fine`, fix plan §9.3a) SGS at 22 m reproduces the fine-grid
  *local-inertial* solution more closely than local-inertial at 22 m does,
  which is what a sub-grid scheme for these equations should do. But the
  fine local-inertial solution itself conveys more than fine SWE in deep,
  fast flow (the momentum physics local-inertial omits), so against the fine
  SWE reference SGS sits low. That is a limit of the local-inertial
  equations, not of the sub-grid representation; whether to add the missing
  momentum losses to SGS is an open decision.

In [ ]:
# What the runs say about the remaining differences (measuring reach, 8 h)
for (job, ref) in (("cart20_sgs_q3", "2d_q3"), ("cart20_sgs_q20", "2d_q20"), ("cart20_sgslumped_q3", "lumped20_q3"),
                   ("cart20_sgslumped_q20", "lumped20_q20"), ("a5r18_sgs_q3", "2d_q3"), ("a5r18_sgs_q20", "a5w_q20"),
                   ("a5r18_std_q20", "a5w_q20"))
    @printf("%-42s δ − analytic %-13s %+.3f m\n", label(job), "(" * ref * ")", m[job]["delta_channel"] - a["delta_$ref"])
end
println("\nCartesian 20 m: the level at which the tables (§4, faces evaluated at the upstream cell's level) carry Q at the bed slope, and the runs")
for Q0 in QS
    q = qtag(Q0)
    @printf("  %2d m³/s: 2-D form, tables %+.3f m, run %+.3f m;  lumped form, tables %+.3f m, run %+.3f m\n", Q0,
            δ_at(F(u["Q"]), F(u["delta"]), Q0), m["cart20_sgs_$q"]["delta_channel"],
            δ_at(F(u["Q_lumped"]), F(u["delta"]), Q0), m["cart20_sgslumped_$q"]["delta_channel"])
end

## 8. Conclusions

- **Storage is measured correctly.** SGS's storage tables hold the terrain's
  volume closely, and overbank the volume above the cell-mean bed equals the
  water volume — for SGS as for the 2.5 m reference. In bank a cell-mean grid
  stores *more* than the true volume (about 2.2× here), the opposite of what
  SGS did at Carlisle. The extra SGS storage of 0.6.0 was real water, held
  back by too little conveyance.
- **Two defects held it back in 0.6.0**, both independent of the storage
  tables: overbank, face flow areas frozen at the cell's highest DEM sample
  and averaged between the two cells (one cross-section of the 20 m strip
  could carry about 5 m³/s at the bed slope, so at 20 m³/s the water piled
  up metres high and was still filling at 8 h); in bank, the legacy donor
  cap, which ran the channel brim-full.
- **What was changed in 0.7.0:** one table per face from its own profile,
  continued above it, with the 2-D conveyance Σ dx·h^(5/3) (`--sgs-conveyance
  lumped` keeps one radius per face); positivity limiting; a time step from
  the hydraulic depth and the faces, with Courant 0.5; and, on A5, the flow
  path between the wetted areas of partly wet cells.
- **Result on the strip:** SGS is steady at the 2-D analytic level within
  about a centimetre on the Cartesian 20 m mesh, in bank and overbank, and
  in bank on A5; overbank on A5 it lies between the 2-D answer for the
  ragged strip's width and local-inertial on the same mesh. The lumped form
  sits at the cell-mean level overbank, as the analysis predicts. All the
  case's checks pass, where eight SGS checks failed in 0.6.0.
- **What remains:** the centimetre low, the cost of the shorter steps on A5,
  the η filter that would let SGS run at Courant 0.7, and, on Carlisle, the
  momentum losses local-inertial lacks in deep, fast flow (notebook 3).

The headline numbers (§1) once more:

In [ ]:
headline